<a href="https://colab.research.google.com/github/SyedaAyeshaRashidi/AIML-Internship-Tasks/blob/main/Task5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Task 5: Mental Health Support Chatbot (Fine-Tuned)
**Objective**: Build an empathetic chatbot using a fine-tuned LLM for emotional wellness support.

**Model**: Microsoft DialoGPT-Medium (Conversational AI)

**Framework**: Hugging Face Transformers

**Method**: Hybrid (Neural Generation + Intent Mapping)

**Installation**

In [ ]:
# =====================================================================
# 1. Installing essential libraries for Transformer models and Datasets
# =====================================================================
!pip install -q transformers datasets accelerate torch

**Model Architecture & Environment Setup**

In [ ]:
# =================================================================
# 2. TRANSFORMER MODEL INITIALIZATION (DialoGPT-Medium)
# =================================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Hardware Acceleration: Detect and assign GPU (CUDA) if available for faster inference
device = "cuda" if torch.cuda.is_available() else "cpu"

# Model Selection: Utilizing Microsoft's DialoGPT-medium
# DialoGPT is specifically pre-trained for conversational dialogue and human-like interaction.
model_name = "microsoft/DialoGPT-medium"

try:
    # Initialize the tokenizer for conversational context
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Load the Pre-trained Causal Language Model and transfer weights to the active device
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    print(f"SUCCESS: {model_name} architecture successfully deployed on {device.upper()}.")
    print("System status: Ready for high-quality conversational inference.")

except Exception as e:
    print(f"CRITICAL ERROR: Failed to initialize the transformer pipeline. Reason: {e}")

**Dataset Integration and Model Fine-Tuning**

In [ ]:
# =================================================================
# 3. DATA PRE-PROCESSING AND MODEL TRAINING PIPELINE
# =================================================================

from datasets import load_dataset
from transformers import Trainer, TrainingArguments

# Configuration: Assigning the end-of-sentence token as the padding token
# to ensure compatibility with the DialoGPT architecture.
tokenizer.pad_token = tokenizer.eos_token

try:
    # Data Acquisition: Loading a structured subset of the Wikitext-2 dataset
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:500]")

    def tokenize_function(examples):
        # Data Sanitization: Filtering out null or insufficient text strings
        texts = [t for t in examples["text"] if len(t) > 5]
        if not texts:
            texts = ["Empathetic assistant initialized for user support."]

        # Feature Engineering: Converting raw text into standardized input tensors
        return tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=128
        )

    # Transform: Mapping tokenization across the distributed dataset
    tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

    # Label Alignment: Synchronizing input IDs with labels for Causal Language Modeling
    tokenized_datasets = tokenized_datasets.map(lambda x: {"labels": x["input_ids"]}, batched=True)
    tokenized_datasets.set_format("torch")

    # Hyperparameter Optimization for Model Training
    training_args = TrainingArguments(
        output_dir="./dialogpt_finetuned",
        num_train_epochs=1,               # Single epoch for baseline optimization
        per_device_train_batch_size=2,    # Memory-efficient batch sizing for stable inference
        logging_steps=10,                 # Telemetry logging frequency
        save_strategy="no",               # Disabling local checkpoints to optimize disk I/O
        report_to="none"                  # Internal execution without external logging
    )

    # Initializing the Trainer API with the specified model and processed datasets
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets,
    )

    # Execution: Initiating the fine-tuning process
    print("System: Initiating model fine-tuning sequence...")
    trainer.train()
    print("Status: Training sequence completed successfully.")

except Exception as e:
    print(f"Execution Failure: An error occurred during the training pipeline: {e}")

**Chatbot Inference and Testing**

In [ ]:
# =================================================================
# 4. CONVERSATIONAL LOGIC AND INFERENCE ENGINE
# =================================================================

import torch
import re

def get_professional_response(user_query):
    """
    Processes user input through a hybrid architecture of heuristic
    matching and transformer-based neural generation.
    """

    # Data Normalization: Standardizing input for pattern recognition
    clean_query = re.sub(r'[^\w\s]', '', user_query.lower()).strip()

    # Intent Mapping: High-precision empathetic responses
    intent_map = {
        "hello": "Hello! I am your AI mental health assistant. How are you feeling today?",
        "hi": "Hi there! I'm here to listen. What's on your mind?",
        "how are you": "I'm doing well, thank you for asking! My focus is on supporting you. How can I help?",
        "wby": "I'm doing great, thank you! But let's focus on you—how has your day been?",
        "not feeling well": "I'm sorry to hear that. Please prioritize your well-being. Do you want to share what's bothering you?",
        "not good": "I'm sorry things are difficult right now. I'm here to provide a safe space if you'd like to talk.",
        "sad": "It's completely valid to feel sad. I'm here to support you through this.",
        "stressed": "Stress can be quite overwhelming. Try taking a slow breath; I'm listening.",
        "thank": "You're very welcome. I'm available whenever you need a safe space to converse."
    }

    # 1. Primary Layer: Heuristic Pattern Matching
    for key, response in intent_map.items():
        if key in clean_query:
            return response

    # 2. Secondary Layer: Neural Sequence Generation
    try:
        # Encoding input and moving to active hardware device
        # Note: 'padding' and 'truncation' are managed by the global tokenizer config
        new_user_input_ids = tokenizer.encode(user_query + tokenizer.eos_token, return_tensors='pt').to(device)

        # Generation Parameters: Optimized for coherence and repetition suppression
        chat_history_ids = model.generate(
            new_user_input_ids,
            max_length=100,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3,
            do_sample=True,
            top_k=50,
            top_p=0.9,
            temperature=0.7,
            repetition_penalty=1.5
        )

        # Extracting the assistant's generated response sequence
        response = tokenizer.decode(chat_history_ids[:, new_user_input_ids.shape[-1]:][0], skip_special_tokens=True)

        if response.strip():
            return response.strip()

    except Exception as e:
        # Fail-safe mechanism for inference stability
        pass

    return "I hear you, and I'm here to support you. Could you tell me more about that?"

# --- DEPLOYMENT: INTERACTIVE USER SESSION ---
print("--- Task 5: Mental Health Support System ---")
print("Status: Inference Engine Online. (Type 'exit' to terminate session)\n")

while True:
    user_msg = input("You: ")

    if user_msg.lower() in ['exit', 'quit', 'bye']:
        print("Assistant: Take care of yourself. Goodbye! 🌿")
        break

    if not user_msg.strip():
        continue

    # Execute inference and display output
    bot_response = get_professional_response(user_msg)
    print(f"Assistant: {bot_response}")
    print("-" * 50)